In [ ]:
asset_path = 'users/your_username/your_asset_name'
degree_size = 0.1
image_collection_id = 'COPERNICUS/S2_HARMONIZED'
bands_to_export = ['B2', 'B3', 'B4']
start_date = '2015-01-01'
end_date = '2026-12-31'
use_csplus = True
csplus_thresh = 0.8
cloud_filter_percentage = 20
export_folder = 'EEExport'
export_scale = 10


In [ ]:
import ee
from ee.feature import Feature
from ee.featurecollection import FeatureCollection
from ee.geometry import Geometry
from ee.imagecollection import ImageCollection
from ee.image import Image
from ee.batch import Export
from ee.filter import Filter

ee.Authenticate()
ee.Initialize(project='ee-yangluhao990714')

In [ ]:
points = FeatureCollection(asset_path)
points_list = points.toList(points.size())
num_points = int(points.size().getInfo() or 0)

for i in range(num_points):
    point_feature = Feature(points_list.get(i))
    point_geometry = point_feature.geometry()
    coords = point_geometry.coordinates().getInfo()
    if coords is None or len(coords) < 2:
        continue
    lon, lat = coords[0], coords[1]
    
    half_size = degree_size / 2.0
    bbox = Geometry.Rectangle([
        lon - half_size, lat - half_size,
        lon + half_size, lat + half_size
    ])

    image_collection = (ImageCollection(image_collection_id)
                        .filterBounds(bbox)
                        .filterDate(start_date, end_date))

    if cloud_filter_percentage is not None:
        image_collection = image_collection.filter(
            Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_filter_percentage)
        )

    if use_csplus:
        cs = ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
        def add_mask(img):
            cs_img = cs.filter(Filter.eq('system:index', img.get('system:index'))).first()
            return img.updateMask(cs_img.select('cs').gt(csplus_thresh))
        image_collection = image_collection.map(add_mask)

    for band_name in bands_to_export:
        time_series_cube = image_collection.select(band_name).toBands()

        collection_name_for_file = image_collection_id.replace('/', '_')
        file_name = f"{collection_name_for_file}_{band_name}_lon{lon:.4f}_lat{lat:.4f}"

        task = Export.image.toDrive(
            image=time_series_cube.toFloat(),
            description=file_name,
            folder=export_folder,
            fileNamePrefix=file_name,
            region=bbox,
            scale=export_scale,
            crs='EPSG:4326',
            maxPixels=1e10
        )
        task.start()
